In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-06 11:03:44--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  --.-KB/s    in 0.07s   

2026-11-06 11:03:44 (336 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# Merge first so we can count by title
df = df_ratings.merge(df_books[["isbn", "title"]], on="isbn")

# Users with >= 200 ratings
user_counts = df["user"].value_counts()
df = df[df["user"].isin(user_counts[user_counts >= 200].index)]

# Then books (by title) with >= 100 ratings among those users
title_counts = df["title"].value_counts()
df = df[df["title"].isin(title_counts[title_counts >= 100].index)]

df = df.drop_duplicates(subset=["title", "user"])

book_user = df.pivot(index="title", columns="user", values="rating").fillna(0)

from scipy.sparse import csr_matrix
matrix = csr_matrix(book_user.values)

model = NearestNeighbors(metric="cosine", algorithm="brute")
model.fit(matrix)

In [8]:
# function to return recommended books - this will be tested
def get_recommends(book=""):
    idx = book_user.index.get_loc(book)
    distances, indices = model.kneighbors(matrix[idx], n_neighbors=6)

    recs = [
        [book_user.index[i], float(d)]
        for i, d in zip(indices[0][1:], distances[0][1:])
    ]
    recs.reverse()  # farthest first, as the test expects
    return [book, recs]

In [13]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# 1. Start with the original ratings dataframe
df = df_ratings.copy()

# 2. Find users who have rated >= 200 books using the raw ratings
user_counts = df['user'].value_counts()
active_users = user_counts[user_counts >= 200].index

# 3. Find books that have >= 100 ratings using the raw ratings
isbn_counts = df['isbn'].value_counts()
popular_isbns = isbn_counts[isbn_counts >= 100].index

# 4. Filter the dataframe to keep only these active users and popular books
df_filtered = df[df['user'].isin(active_users) & df['isbn'].isin(popular_isbns)]

# 5. Merge with books dataframe to get the titles
df_merged = df_filtered.merge(df_books, on='isbn')

# 6. Drop duplicates of the same user rating the same book title
df_clean = df_merged.drop_duplicates(subset=['title', 'user'])

# 7. Pivot to create the user-item matrix
book_user = df_clean.pivot(index='title', columns='user', values='rating').fillna(0)

# 8. Create the sparse matrix representation
matrix = csr_matrix(book_user.values)

# 9. Train the Nearest Neighbors model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(matrix)

# 10. Define the recommendation function exactly as expected
def get_recommends(book=""):
    if book not in book_user.index:
        return [book, []]

    idx = book_user.index.get_loc(book)
    distances, indices = model.kneighbors(matrix[idx], n_neighbors=6)

    # The test expects 4 specific recommendations. We extract neighbors 1 to 5
    recs = []
    for i in range(1, 6):
        recs.append([book_user.index[indices[0][i]], float(distances[0][i])])

    # The challenge test asserts: recommends[1][0] and recommends[1][1]
    # are among the expected list, sorted in a specific order (farthest distance first)
    recs.reverse()
    return [book, recs]

# Run the tests
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge!")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]
You passed the challenge!
